# 🏆 SHL Grammar Scoring Engine 2026

**Goal:** Predict a continuous MOS Likert grammar score (0–5) from 45–60s spoken WAV files.  
**Strategy:** Three-Tower Ensemble → Whisper ASR + DeBERTa-v3-base · HuBERT-large MLP · LightGBM  
**Metric:** RMSE ↓ · Pearson ↑  
**Target:** RMSE < 0.32 (beat leaderboard rank 1 at 0.3239)

```
Dataset_Final/
├── train/   (769 .wav files)
├── test/    (216 .wav files)
├── train.csv           → filename, label
├── test.csv            → filename
└── sample_submission.csv
```

## ① Install Dependencies

In [1]:
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("openai-whisper", "language-tool-python", "lightgbm", "librosa", "soundfile",
    "transformers[torch]", "accelerate", "sentencepiece", "scipy", "tqdm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 3.1 MB/s eta 0:00:00


## ② Imports & Paths

In [2]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import whisper
import librosa
import language_tool_python
import lightgbm as lgb
from pathlib import Path
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Wav2Vec2FeatureExtractor, HubertModel,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

warnings.filterwarnings("ignore")
SEED   = 42
N_FOLD = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device : {DEVICE}")
if DEVICE == "cuda": print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── Exact dataset paths ────────────────────────────────────────────────────────
ROOT       = Path("/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final")
TRAIN_AUDIO= ROOT / "train"          # 769 wav files
TEST_AUDIO = ROOT / "test"           # 216 wav files
WORK_DIR   = Path("/kaggle/working")

assert ROOT.exists(),        f"Root not found: {ROOT}"
assert TRAIN_AUDIO.exists(), f"Train audio dir not found: {TRAIN_AUDIO}"
assert TEST_AUDIO.exists(),  f"Test audio dir not found: {TEST_AUDIO}"
print(f"Train WAVs : {len(list(TRAIN_AUDIO.glob('*.wav')))}")
print(f"Test  WAVs : {len(list(TEST_AUDIO.glob('*.wav')))}")

Device : cuda
GPU    : Tesla T4
Train WAVs : 769
Test  WAVs : 216


## ③ Load CSVs & Inspect

In [3]:
train_df = pd.read_csv(ROOT / "train.csv")
test_df  = pd.read_csv(ROOT / "test.csv")
sample   = pd.read_csv(ROOT / "sample_submission.csv")

print("train.csv columns :", train_df.columns.tolist())
print("test.csv  columns :", test_df.columns.tolist())
print("sample_submission :", sample.columns.tolist())
print()
print(train_df.head(3))
print()
print(f"Label range : [{train_df.iloc[:,1].min():.2f}, {train_df.iloc[:,1].max():.2f}]")
print(f"Label mean  : {train_df.iloc[:,1].mean():.3f} ± {train_df.iloc[:,1].std():.3f}")

# Auto-detect column names from sample_submission
FILE_COL  = sample.columns[0]   # e.g. 'filename'
SCORE_COL = sample.columns[1]   # e.g. 'label'
LABEL_COL = train_df.columns[1] # label column in train.csv
print(f"\nFILE_COL={FILE_COL!r}  SCORE_COL={SCORE_COL!r}  LABEL_COL={LABEL_COL!r}")

def resolve_wav(fname, split="train"):
    """Return full path to wav; handles filenames with/without .wav extension."""
    audio_dir = TRAIN_AUDIO if split == "train" else TEST_AUDIO
    f = str(fname)
    p = audio_dir / (f if f.endswith(".wav") else f + ".wav")
    return p

train.csv columns : ['filename', 'label']
test.csv  columns : ['filename', 'label']
sample_submission : ['filename', 'label']

        filename  label
0  audio_192.wav    3.0
1  audio_436.wav    3.0
2  audio_447.wav    3.5

Label range : [0.00, 5.00]
Label mean  : 3.313 ± 1.239

FILE_COL='filename'  SCORE_COL='label'  LABEL_COL='label'


## ④ Tower A — ASR: Whisper large-v3
State-of-the-art open-source ASR. Results cached to disk after first run.

In [4]:
TRANS_CACHE = WORK_DIR / "transcriptions.csv"

if TRANS_CACHE.exists():
    trans_df = pd.read_csv(TRANS_CACHE)
    print(f"✓ Loaded cached transcriptions ({len(trans_df)} rows)")
else:
    print("Loading Whisper large-v3 …")
    wmodel = whisper.load_model("large-v3", device=DEVICE)

    rows = []
    for split, df in [("train", train_df), ("test", test_df)]:
        for fname in tqdm(df[FILE_COL], desc=f"Whisper [{split}]"):
            path = resolve_wav(fname, split)
            try:
                r = wmodel.transcribe(
                    str(path), fp16=(DEVICE=="cuda"), language="en",
                    condition_on_previous_text=False, temperature=0
                )
                text = r["text"].strip()
            except Exception as e:
                print(f"  ✗ {fname}: {e}")
                text = ""
            rows.append({FILE_COL: fname, "split": split, "transcript": text})

    trans_df = pd.DataFrame(rows)
    trans_df.to_csv(TRANS_CACHE, index=False)
    del wmodel; torch.cuda.empty_cache()
    print(f"✓ Transcription done → {TRANS_CACHE}")

train_trans = trans_df[trans_df["split"]=="train"][[FILE_COL, "transcript"]]
test_trans  = trans_df[trans_df["split"]=="test"][[FILE_COL, "transcript"]]

train_df = train_df.merge(train_trans, on=FILE_COL, how="left")
test_df  = test_df.merge(test_trans,   on=FILE_COL, how="left")
train_df["transcript"] = train_df["transcript"].fillna("").astype(str)
test_df["transcript"]  = test_df["transcript"].fillna("").astype(str)

print("Sample train transcript:", train_df["transcript"].iloc[0][:120])

Loading Whisper large-v3 …


100%|██████████████████████████████████████| 2.88G/2.88G [00:10<00:00, 288MiB/s]


Whisper [train]:   0%|          | 0/769 [00:00<?, ?it/s]

Whisper [test]:   0%|          | 0/216 [00:00<?, ?it/s]

✓ Transcription done → /kaggle/working/transcriptions.csv
Sample train transcript: The playground is very, very beautiful and fun. There are playtools like the merry-go-round, the swing. The students usu


## ⑤ WavLM-large Audio Embeddings
Extracts 1024-dim acoustic embeddings using WavLM (SOTA for speech).

In [5]:
WAVLM_CACHE = WORK_DIR / "wavlm_embeddings.npy"
WAVLM_META  = WORK_DIR / "wavlm_meta.json"

if WAVLM_CACHE.exists():
    emb_all   = np.load(WAVLM_CACHE)
    meta      = json.load(open(WAVLM_META))
    n_tr      = meta["n_train"]
    print(f"✓ Loaded cached WavLM embeddings {emb_all.shape}")
else:
    from transformers import Wav2Vec2FeatureExtractor, WavLMModel
    print("Loading WavLM-large …")
    feat_ext = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-large")
    wavlm    = WavLMModel.from_pretrained("microsoft/wavlm-large").to(DEVICE)
    wavlm.eval()

    def embed_wav(path):
        try:
            y, _ = librosa.load(str(path), sr=16000, mono=True)
            y    = y[:16000*62] # cap at 62 s
            inp  = feat_ext(y, sampling_rate=16000, return_tensors="pt", padding=True)
            inp  = {k: v.to(DEVICE) for k, v in inp.items()}
            with torch.no_grad():
                out = wavlm(**inp).last_hidden_state
            return out.mean(dim=1).squeeze().cpu().numpy()
        except Exception as e:
            print(f"  ✗ {path.name}: {e}")
            return np.zeros(1024)

    tr_embs = [embed_wav(resolve_wav(f, "train")) for f in tqdm(train_df[FILE_COL], desc="WavLM [train]")]
    te_embs = [embed_wav(resolve_wav(f, "test"))  for f in tqdm(test_df[FILE_COL],  desc="WavLM [test]")]

    n_tr    = len(tr_embs)
    emb_all = np.vstack(tr_embs + te_embs)
    np.save(WAVLM_CACHE, emb_all)
    json.dump({"n_train": n_tr}, open(WAVLM_META, "w"))
    del wavlm; torch.cuda.empty_cache()
    print(f"✓ WavLM done → {emb_all.shape}")

wavlm_train = emb_all[:n_tr]
wavlm_test  = emb_all[n_tr:]


Loading WavLM-large …


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.22k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.26GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

WavLM [train]:   0%|          | 0/769 [00:00<?, ?it/s]

WavLM [test]:   0%|          | 0/216 [00:00<?, ?it/s]

✓ WavLM done → (985, 1024)


## ⑥ BGE-Large Text Embeddings
Using BAAI/bge-large-en-v1.5 for high-quality text representation.

In [6]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

print("Loading SentenceTransformer (BAAI/bge-large-en-v1.5)...")
st_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=DEVICE)

print("Extracting text embeddings...")
text_emb_train = st_model.encode(train_df["transcript"].tolist(), show_progress_bar=True)
text_emb_test  = st_model.encode(test_df["transcript"].tolist(), show_progress_bar=True)
del st_model; torch.cuda.empty_cache()


Loading SentenceTransformer (BAAI/bge-large-en-v1.5)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Extracting text embeddings...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

## ⑦ Handcrafted Features
LanguageTool errors and basic linguistic features.

In [7]:
FEAT_CACHE = WORK_DIR / "handcrafted_features.npy"

if FEAT_CACHE.exists():
    feat_all = np.load(FEAT_CACHE)
    print(f"✓ Loaded cached features {feat_all.shape}")
else:
    lt = language_tool_python.LanguageTool("en-US")
    def text_feats(text):
        text  = str(text).strip()
        if not text: return np.zeros(10)
        words = text.split()
        n     = max(len(words), 1)
        sents = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        matches = lt.check(text)
        n_err   = len(matches)
        gram_e, spell_e = 0, 0
        for m in matches:
            rule_str = str(getattr(m, "rule_id", getattr(m, "ruleId", getattr(m, "category", "")))).upper()
            if any(k in rule_str for k in ["GRAMMAR", "TYPO", "SYNTAX"]): gram_e += 1
            if any(k in rule_str for k in ["SPELL", "MORFOLOGIK"]): spell_e += 1
        punct   = sum(1 for c in text if c in ".,;:!?")
        caps    = sum(1 for w in words if w and w[0].isupper())
        unique  = len(set(w.lower() for w in words))
        return np.array([
            n_err / n, gram_e / n, spell_e / n,
            n, len(sents),
            np.mean([len(w) for w in words]),
            unique / n, unique,
            punct / n, caps / n,
        ], dtype=float)

    print("Extracting text features …")
    tr_txt = np.stack([text_feats(t) for t in tqdm(train_df["transcript"])])
    te_txt = np.stack([text_feats(t) for t in tqdm(test_df["transcript"])])

    # Skipping audio handcrafted to save time and rely on WavLM
    lt.close()
    feat_all = np.vstack([tr_txt, te_txt])
    np.save(FEAT_CACHE, feat_all)
    print(f"✓ Features done → {feat_all.shape}")

hand_train = feat_all[:len(train_df)]
hand_test  = feat_all[len(train_df):]


Extracting text features …


  0%|          | 0/769 [00:00<?, ?it/s]

  0%|          | 0/216 [00:00<?, ?it/s]

✓ Features done → (985, 10)


## ⑧ Final Stacking Ensemble
Using sklearn's StackingRegressor with WavLM + BGE + Handcrafted features.

In [8]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge, BayesianRidge, RidgeCV
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA

# Concatenate all features
X_train = np.hstack([hand_train, wavlm_train, text_emb_train])
X_test  = np.hstack([hand_test,  wavlm_test,  text_emb_test])
y_train = train_df[LABEL_COL].values.astype(float)
print(f"Total Feature Matrix Shape: {X_train.shape}")

# Scale the features
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc  = sc.transform(X_test)

# Define Level-0 estimators
estimators = [
    ('ridge', Ridge(alpha=100.0)),
    ('bayes', BayesianRidge()),
    ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1)),
    ('lgbm', lgb.LGBMRegressor(n_estimators=500, learning_rate=0.01, num_leaves=15, subsample=0.8, colsample_bytree=0.5, random_state=SEED, verbose=-1))
]

# Define Stacking Regressor
stack = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]),
    cv=5,
    n_jobs=-1
)

print("Training Stacking Regressor with 5-Fold CV...")
stack.fit(X_train_sc, y_train)

test_preds = np.clip(stack.predict(X_test_sc), 0, 5)

# Calculate Training RMSE as a quick sanity check (OOF is handled internally by StackingRegressor)
train_preds = np.clip(stack.predict(X_train_sc), 0, 5)
train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
train_pearson = pearsonr(y_train, train_preds)[0]
print(f"\nTrain RMSE: {train_rmse:.4f}  |  Train Pearson: {train_pearson:.4f}")


Total Feature Matrix Shape: (769, 2058)
Training Stacking Regressor with 5-Fold CV...


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Train RMSE: 0.1908  |  Train Pearson: 0.9884


## ⑨ Generate Submission

In [9]:
submission = pd.DataFrame({
    FILE_COL  : test_df[FILE_COL].values,
    SCORE_COL : test_preds,
})

OUT = WORK_DIR / "submission.csv"
submission.to_csv(OUT, index=False)
print(f"✅ NEW submission.csv saved to {OUT}")
print(submission.head(10).to_string(index=False))


✅ NEW submission.csv saved to /kaggle/working/submission.csv
     filename    label
audio_128.wav 2.956917
audio_156.wav 3.590825
 audio_19.wav 4.854571
audio_108.wav 2.251055
audio_206.wav 2.475609
audio_161.wav 3.129439
audio_215.wav 3.100474
audio_213.wav 2.278987
 audio_11.wav 3.016645
audio_155.wav 3.013473
